In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm
from collections import defaultdict

In [2]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

device = utils.device

In [3]:
def load_sample(data_name, user_type, flag_replace_NER):
    N_icl = [1, 3]

    from src.data_loader import Loader
    loader = Loader(dir_workspace, version_exp, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER)
    dict_data = loader.load_data()

    dict_flag = dict_data["flag"]

    dict_text = dict_data["profile"]
    for text_type in ["concat"]:
        dict_text.update(dict_data[text_type])

    d_documents = dict_data["items"]["candidates"]

    users = list(dict_flag[user_type.replace("concat_", "")].keys())
    user = users[0]

    user_text = dict_text[user_type][user]
    print("--" * 10, "user text", "--" * 10)
    print(user_text)
    print()

    s_flag = pd.Series(dict_flag[user_type.replace("concat_", "")][user])
    df_candidates = pd.DataFrame({
        "item_text": pd.Series({item: d_documents[item] for item in s_flag.index}),
        "flag": s_flag
    }).iloc[:5]

    print("--" * 10, "item text", "--" * 10)
    print(df_candidates["item_text"].iloc[0])

    return user_text, df_candidates, s_flag, d_documents

In [4]:
user_text, df_candidates, s_flag, d_documents = load_sample(
    data_name="Job",
    user_type='profile_mid-career',
    flag_replace_NER=False
)

-------------------- user text --------------------
{'degree type': "Bachelor's", 'major': 'Economics', 'graduation year': 2001, 'work history count': 3, 'total years experience': 9, 'currently employed': 'Yes', 'managed others': 'No', 'managed how many': 0, 'work history': {'1': 'Customer Service Representative', '2': 'Amenities Attendant', '3': 'Dietary Assistant'}}

-------------------- item text --------------------
title : Data Entry Clerk
description : The Mergis Group
is a staffing and recruitment industry leader with thousands of satisfied clients nationwide. We offer the successful candidate the opportunity to work for our clients in either full-time or temporary positions. We pride ourselves on our strong commitment to client satisfaction, and our focus on helping our employees find their next job.
We are working with one of our partner companies seeking a
Data Entry Clerk in the Bill Auditing department to complete a 3-6 month assignment.
Bill Auditor
Description
Reviews bil

In [5]:
# load embedding model
model_name_emb = "Qwen/Qwen3-Embedding-0.6B"
from src.embedding import Embedding
model_id = f"{dir_parent}/models/embedding_models/{model_name_emb}"
emb = Embedding(model_id, device)

# embed user text
v_user = emb.encode(user_text, query=True)

# embed candidate items
V_candidates = pd.DataFrame({
    item: emb.encode(item_text, query=False)
    for item, item_text in df_candidates["item_text"].to_dict().items()
}).T

# compute similarity
from sklearn.metrics.pairwise import cosine_similarity
s_sim = pd.Series(
    cosine_similarity([v_user], V_candidates.values)[0],
    index=V_candidates.index
)

# add similarity and sort
df_ = df_candidates.copy()
df_ = pd.concat([df_, pd.DataFrame({"score": s_sim})], axis=1)
df_ = df_.sort_values(by="score", ascending=False)
display(df_)

Loading weights: 100%|███████████████████████████████████████████| 310/310 [00:02<00:00, 109.18it/s, Materializing param=norm.weight]


,item_text,flag,score
I10312,title : Receptionist\ndescription : To assist ...,0,0.390662
I1105002,title : Customer Service Representative\ndescr...,0,0.385450
I1051924,title : Customer Support Representative TN\nd...,0,0.370723
I1089723,title : Data Entry Clerk\ndescription : The Me...,1,0.366144
I7084,title : Research Assistant\ndescription : Otte...,1,0.336558


In [6]:
# LLM
model_name_llm = "gpt-4.1-mini-2025-04-14"

candidate_size = 10
at_K = 5
from src.reranker_llm import LLMReranker
llmreranker = LLMReranker(candidate_size=candidate_size)

llm = utils.load_llm(model_name=model_name_llm)
llmreranker.fit(llm)
items, flags, d_candidates = llmreranker._prepare(s_flag, d_documents)
prompt = llmreranker._prompt(user_text, d_candidates)

print("--" * 10, "prompt", "--" * 10)
print(prompt)

-------------------- prompt --------------------
# Task
Your task is to recommend exactly 10 items from the provided candidate set, ordered from most to least likely to be preferred by the user.

# Constraints
- Select items only from the provided candidate set; do not invent new items or IDs.
- Do not include any items the user has already interacted with.
- Return only a Python list literal of exactly 10 unique integer item IDs (the keys of the candidate set), ordered by preference, e.g., [8, 4, ...]. Do not output anything else.
- Make the ranking deterministic; if items are equally relevant, break ties by ascending item ID.
- Base your ranking only on the information in this prompt (user history and candidate metadata).

# Data
User Information:
{'degree type': "Bachelor's", 'major': 'Economics', 'graduation year': 2001, 'work history count': 3, 'total years experience': 9, 'currently employed': 'Yes', 'managed others': 'No', 'managed how many': 0, 'work history': {'1': 'Customer S

In [7]:
user_text, df_candidates, s_flag, d_documents = load_sample(
    data_name="Job",
    user_type='profile_mid-career',
    flag_replace_NER=True
)

-------------------- user text --------------------
{'degree type': "Bachelor's", 'major': 'Economics', 'graduation year': <DATE>, 'work history count': <CARDINAL>, 'total <DATE> experience': <CARDINAL>, 'currently employed': 'Yes', 'managed others': 'No', 'managed how many': <CARDINAL>, 'work history': {<DATE>: 'Customer Service Representative', '2': 'Amenities Attendant', '3': 'Dietary Assistant'}}

-------------------- item text --------------------
title : Data Entry Clerk
description : <ORG>
is a staffing and recruitment industry leader with <CARDINAL> of satisfied clients nationwide. We offer the successful candidate the opportunity to work for our clients in either full-time or temporary positions. We pride ourselves on our strong commitment to client satisfaction, and our focus on helping our employees find their next job.
We are working with one of our partner companies seeking a
Data Entry Clerk in the <ORG> department to complete a <DATE> assignment.
<PERSON> bills submitted

In [8]:
for f in [False, True]:
    print("==" * 20, f"NER flag: {f}", "==" * 20)
    user_text, df_candidates, s_flag, d_documents = load_sample(
        data_name="MovieLens",
        user_type='profile',
        flag_replace_NER=f
    )

======================================== NER flag: False ========================================
-------------------- user text --------------------
{'gender': 'M', 'age': '50-55', 'occupation': 'lawyer'}

-------------------- item text --------------------
title : Stardust Memories (1980)
genres : comedy, drama
======================================== NER flag: True ========================================
-------------------- user text --------------------
{'gender': 'M', 'age': '<CARDINAL>', 'occupation': 'lawyer'}

-------------------- item text --------------------
title : <ORG> (<DATE>)
genres : comedy, drama


In [9]:
for f in [False, True]:
    print("==" * 20, f"NER flag: {f}", "==" * 20)
    user_text, df_candidates, s_flag, d_documents = load_sample(
        data_name="ARD_CDs_and_Vinyl",
        user_type='concat_3-sample',
        flag_replace_NER=f
    )

======================================== NER flag: False ========================================
-------------------- user text --------------------
#log 1
title : The Best of Pink Floyd - A Foot In The Door
category : rock, progressive, progressive rock
description : 2011 EMI edition. Remastered 16-track compilation featuring newly created artwork from Storm Thorgerson.
#log 2
title : The Best of Bachman-Turner Overdrive: The Millennium Collection 20th Century Masters
category : today's deals in music, classic rock, album-oriented rock (aor), cds $7 - $10
description : Tracks: "Gimme Your Money Please," "Let It Ride," "Blue Collar," "Takin Care of Business," "You Ain't Seen Nothin Yet," "Roll on Down the Highway" "Hey You," "Take It Like a Man," "Lookin Out for #1," "Shotgun Rider," "I'm in Love" & "Heartaches."
#log 3
title : Damn Country Music
category : country, today's country
description : DAMN COUNTRY MUSIC will be his third album from Big Machine Records with its first single 

In [10]:
for f in [False, True]:
    print("==" * 20, f"NER flag: {f}", "==" * 20)
    user_text, df_candidates, s_flag, d_documents = load_sample(
        data_name="ARD_Toys_and_Games",
        user_type='concat_3-sample',
        flag_replace_NER=f
    )

======================================== NER flag: False ========================================
-------------------- user text --------------------
#log 1
title : Frozen Olaf's in Trouble Game
category : games & accessories, board games
description : Whether via a game board with dice, a deck of cards, simple lines drawn on scrap paper, or electronic media, gaming is a global pastime that has enriched culture for millennia. From the most classic tabletop board games to up-and-active, play-to-learn games for preschoolers to the painfully funny party games that satisfy your wild side, Hasbro Gaming is a one-stop-shop for filling your games closet.  While continuing to produce some of the most memorable games in the history of family gaming, Hasbro Gaming stays up-and-coming by developing games that incorporate digital content and by partnering with some of the biggest names in entertainment.  Hasbro Gaming and all related properties and characters are trademarks of Hasbro.
#log 2
title